In [8]:
def reverse_image_sequence_numbering(input_folder, output_folder=None):
    """
    Reverses the numbering order of image sequences in a folder.
    Images grouped by the first 4 name elements will have their numbers reversed.
    For example: C_0 becomes C_44, C_1 becomes C_43, etc.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_folder : str, optional
        Path to save the renamed images. If None, creates a 'Reversed' subfolder
        in the input folder (default: None).

    Returns
    -------
    None
    """
    import os
    import shutil
    from collections import defaultdict

    # Set output folder
    if output_folder is None:
        output_folder = os.path.join(input_folder, "Reversed")
    
    os.makedirs(output_folder, exist_ok=True)

    # Dictionary to group images by the first four name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):
            # Extract the first four name elements as the group key
            group_key = "_".join(file_name.split("_")[:4])
            grouped_images[group_key].append(file_name)

    # Process each group
    for group_key, file_names in grouped_images.items():
        # Sort files numerically by channel number
        def extract_channel_number(filename):
            try:
                return int(filename.split("_")[-1].split(".")[0])
            except (ValueError, IndexError):
                return 0
        
        file_names.sort(key=extract_channel_number)
        
        # Get the max number in the sequence
        max_number = extract_channel_number(file_names[-1])
        
        print(f"\nProcessing group: {group_key}")
        print(f"  Sequence range: 0 to {max_number}")
        print(f"  Total images: {len(file_names)}")
        
        # Reverse the numbering
        for file_name in file_names:
            # Get original number and file extension
            parts = file_name.rsplit("_", 1)
            base_name = parts[0]  # Everything before the last underscore
            last_part = parts[1]  # e.g., "0.tif"
            
            original_number = extract_channel_number(file_name)
            extension = "." + last_part.split(".", 1)[1]  # Get file extension
            
            # Calculate reversed number
            reversed_number = max_number - original_number
            
            # Create new filename with reversed number
            new_file_name = f"{base_name}_{reversed_number}{extension}"
            
            # Copy file with new name to output folder
            src_path = os.path.join(input_folder, file_name)
            dst_path = os.path.join(output_folder, new_file_name)
            shutil.copy2(src_path, dst_path)
            
            print(f"  {file_name} → {new_file_name}")
    
    print(f"\nReversed images saved to: {output_folder}")


# Example usage:
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All'
reverse_image_sequence_numbering(input_folder)


Processing group: Tri_10uMw3_03_C
  Sequence range: 0 to 44
  Total images: 45
  Tri_10uMw3_03_C_0.tiff → Tri_10uMw3_03_C_44.tiff
  Tri_10uMw3_03_C_1.tiff → Tri_10uMw3_03_C_43.tiff
  Tri_10uMw3_03_C_2.tiff → Tri_10uMw3_03_C_42.tiff
  Tri_10uMw3_03_C_3.tiff → Tri_10uMw3_03_C_41.tiff
  Tri_10uMw3_03_C_4.tiff → Tri_10uMw3_03_C_40.tiff
  Tri_10uMw3_03_C_5.tiff → Tri_10uMw3_03_C_39.tiff
  Tri_10uMw3_03_C_6.tiff → Tri_10uMw3_03_C_38.tiff
  Tri_10uMw3_03_C_7.tiff → Tri_10uMw3_03_C_37.tiff
  Tri_10uMw3_03_C_8.tiff → Tri_10uMw3_03_C_36.tiff
  Tri_10uMw3_03_C_9.tiff → Tri_10uMw3_03_C_35.tiff
  Tri_10uMw3_03_C_10.tiff → Tri_10uMw3_03_C_34.tiff
  Tri_10uMw3_03_C_11.tiff → Tri_10uMw3_03_C_33.tiff
  Tri_10uMw3_03_C_12.tiff → Tri_10uMw3_03_C_32.tiff
  Tri_10uMw3_03_C_13.tiff → Tri_10uMw3_03_C_31.tiff
  Tri_10uMw3_03_C_14.tiff → Tri_10uMw3_03_C_30.tiff
  Tri_10uMw3_03_C_15.tiff → Tri_10uMw3_03_C_29.tiff
  Tri_10uMw3_03_C_16.tiff → Tri_10uMw3_03_C_28.tiff
  Tri_10uMw3_03_C_17.tiff → Tri_10uMw3_03_C_27

In [3]:
import os
import cv2
import pandas as pd
from collections import defaultdict

def calculate_mean_gray_first_last(input_folder, output_excel_path):
    """
    Calculates the mean gray value of the first and last image in a sequence of images
    that share the same first three name elements (separated by underscores). Also calculates
    the change in intensity (difference and percentage change). Saves the results to an Excel file.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Dictionary to group images by the first three name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first three name elements as the group key
            group_key = "_".join(file_name.split("_")[:4])
            grouped_images[group_key].append(file_name)

    # List to store results
    results = []

    # Process each group
    for group_key, file_names in grouped_images.items():
        # Sort the file names numerically by the channel number (C_X)
        def extract_channel_number(filename):
            """Extract the numeric channel number from filename like Co_10uMw3_01_C_0"""
            try:
                # Split by underscore and get the last part (e.g., "0" from "C_0")
                return int(filename.split("_")[-1].split(".")[0])
            except (ValueError, IndexError):
                return 0
        
        file_names.sort(key=extract_channel_number)

        # Get the first and last image in the group
        first_image_path = os.path.join(input_folder, file_names[0])
        last_image_path = os.path.join(input_folder, file_names[-1])

        # Load the first image and calculate its mean gray value
        first_image = cv2.imread(first_image_path, cv2.IMREAD_GRAYSCALE)
        if first_image is not None:
            first_mean_gray = first_image.mean()
        else:
            print(f"Error: Unable to load image at {first_image_path}")
            first_mean_gray = None

        # Load the last image and calculate its mean gray value
        last_image = cv2.imread(last_image_path, cv2.IMREAD_GRAYSCALE)
        if last_image is not None:
            last_mean_gray = last_image.mean()
        else:
            print(f"Error: Unable to load image at {last_image_path}")
            last_mean_gray = None

        # Calculate the change in intensity (difference and percentage change)
        if first_mean_gray is not None and last_mean_gray is not None:
            change_in_intensity = last_mean_gray - first_mean_gray
            percentage_change = ((last_mean_gray - first_mean_gray) / first_mean_gray) * 100 if first_mean_gray != 0 else None
        else:
            change_in_intensity = None
            percentage_change = None

        # Append the results for this group
        results.append({
            "Group Key": group_key,
            "First Image": file_names[0],
            "First Mean Gray Value": first_mean_gray,
            "Last Image": file_names[-1],
            "Last Mean Gray Value": last_mean_gray,
            "Change in Intensity": change_in_intensity,
            "Percentage Change (%)": percentage_change
        })

    # Convert results to a DataFrame
    df = pd.DataFrame(results)

    # Save the results to an Excel file
    df.to_excel(output_excel_path, index=False)
    print(f"Results saved to Excel file: {output_excel_path}")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All'
output_excel_path = os.path.join(input_folder, "0_Mean_Gray_Values_First_Last.xlsx")
calculate_mean_gray_first_last(input_folder, output_excel_path)

Results saved to Excel file: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/0_Mean_Gray_Values_First_Last.xlsx


In [9]:
import os
import cv2
import pandas as pd
from collections import defaultdict
import re

def calculate_mean_gray_all_images(input_folder, output_excel_path):
    """
    Calculates the mean gray value of every image in the folder.
    Saves the results to an Excel file with separate sheets for each sample
    (grouped by first 2 elements of filename).
    Within each sheet, replicates are displayed in side-by-side columns
    separated by empty columns.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Dictionary to group images by the first 2 name elements
    grouped_images = defaultdict(list)

    # Get all image files
    image_files = [f for f in os.listdir(input_folder) 
                   if f.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg"))]
    
    print(f"Found {len(image_files)} image files")
    
    # Group files by first 2 elements (e.g., "Co_10uMw3")
    for file_name in image_files:
        # Extract the first 2 name elements as the group key
        parts = file_name.split("_")
        if len(parts) >= 2:
            group_key = "_".join(parts[:2])
            grouped_images[group_key].append(file_name)
        else:
            # If filename doesn't have enough underscores, use full name
            grouped_images[file_name].append(file_name)
    
    print(f"Grouped into {len(grouped_images)} sample groups")

    # Create Excel writer object
    with pd.ExcelWriter(output_excel_path, engine='openpyxl') as writer:
        total_images = 0
        
        # Process each group
        for group_key in sorted(grouped_images.keys()):
            file_names = grouped_images[group_key]
            
            # Further subdivide by replicate (3rd element)
            replicate_groups = defaultdict(list)
            for file_name in file_names:
                parts = file_name.split("_")
                if len(parts) >= 3:
                    replicate_id = parts[2]  # e.g., "01", "02", "03"
                    replicate_groups[replicate_id].append(file_name)
                else:
                    replicate_groups["unknown"].append(file_name)
            
            # Dictionary to store data for each replicate
            replicate_data = {}
            max_rows = 0
            
            # Process each replicate
            for replicate_id in sorted(replicate_groups.keys()):
                replicate_files = replicate_groups[replicate_id]
                replicate_files.sort()  # Sort files within each replicate
                
                rep_rows = []
                mean_gray_values = []  # Store mean gray values for averaging
                
                # Process each image in the replicate
                for file_name in replicate_files:
                    image_path = os.path.join(input_folder, file_name)
                    
                    # Load the image and calculate its mean gray value
                    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                    
                    if image is not None:
                        mean_gray = image.mean()
                        mean_gray_values.append(mean_gray)
                        rep_rows.append([file_name, mean_gray])
                    else:
                        print(f"Error: Unable to load image at {image_path}")
                        rep_rows.append([file_name, None])
                
                # Calculate averages of images 30-32 (indices 29-31) and last 3 images
                avg_images_30_32 = None
                avg_last_3 = None
                change_in_intensity = None
                percentage_change = None
                
                # Check if we have at least 32 images for the 30-32 range
                if len(mean_gray_values) >= 32:
                    avg_images_30_32 = sum(mean_gray_values[29:32]) / 3  # indices 29, 30, 31
                
                if len(mean_gray_values) >= 3:
                    avg_last_3 = sum(mean_gray_values[-3:]) / 3
                
                # Calculate change metrics if both averages exist
                if avg_images_30_32 is not None and avg_last_3 is not None:
                    change_in_intensity = avg_last_3 - avg_images_30_32
                    if avg_images_30_32 != 0:
                        percentage_change = (change_in_intensity / avg_images_30_32) * 100
                
                # Add blank row separator
                rep_rows.append(["", None])
                
                # Add summary rows for this replicate
                if avg_images_30_32 is not None:
                    rep_rows.append([f"AVERAGE - Images 30-32", avg_images_30_32])
                if avg_last_3 is not None:
                    rep_rows.append([f"AVERAGE - Last 3", avg_last_3])
                if change_in_intensity is not None:
                    rep_rows.append([f"CHANGE in Intensity", change_in_intensity])
                if percentage_change is not None:
                    rep_rows.append([f"PERCENTAGE Change", percentage_change])
                
                # Store this replicate's data
                replicate_data[replicate_id] = rep_rows
                max_rows = max(max_rows, len(rep_rows))
                
                total_images += len(replicate_files)
                print(f"  Replicate {replicate_id}: {len(replicate_files)} images")
                if avg_images_30_32 is not None:
                    print(f"    Average images 30-32: {avg_images_30_32:.2f}")
                if avg_last_3 is not None:
                    print(f"    Average last 3: {avg_last_3:.2f}")
                if change_in_intensity is not None:
                    print(f"    Change in intensity: {change_in_intensity:.2f}")
                if percentage_change is not None:
                    print(f"    Percentage change: {percentage_change:.2f}%")
            
            # Build side-by-side columns with empty column separators
            combined_data = {}
            
            for idx, replicate_id in enumerate(sorted(replicate_data.keys())):
                # Column headers for this replicate
                col_filename = f"Replicate {replicate_id} - Filename"
                col_value = f"Replicate {replicate_id} - Mean Gray"
                
                # Pad rows to max_rows length
                rep_rows = replicate_data[replicate_id]
                while len(rep_rows) < max_rows:
                    rep_rows.append(["", None])
                
                # Add to combined data
                combined_data[col_filename] = [row[0] for row in rep_rows]
                combined_data[col_value] = [row[1] for row in rep_rows]
                
                # Add empty column separator (except after last replicate)
                if idx < len(replicate_data) - 1:
                    combined_data[f"_spacer_{idx}"] = [""] * max_rows
            
            # Convert to DataFrame
            df = pd.DataFrame(combined_data)
            
            # Sanitize sheet name more aggressively
            sheet_name = group_key[:31]  # Excel max sheet name length is 31
            # Remove ALL special characters that Excel doesn't like
            sheet_name = re.sub(r'[\\/*?:\[\]]', '_', sheet_name)
            # Remove any control characters or non-printable characters
            sheet_name = ''.join(char for char in sheet_name if char.isprintable())
            # Ensure sheet name is not empty
            if not sheet_name:
                sheet_name = "Sheet"
            
            # Write to Excel sheet
            try:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet '{sheet_name}': {len(replicate_groups)} replicates")
            except Exception as e:
                print(f"Error writing sheet '{sheet_name}': {e}")

    print(f"\nResults saved to Excel file: {output_excel_path}")
    print(f"Total sheets created: {len(grouped_images)}")
    print(f"Total images processed: {total_images}")

# Example usage
input_folder = '/Users/allisonpickle/Library/CloudStorage/Box-Box/0_Şeker Lab/0_Experiments/260122_Calcium Imaging_w3/All/Reversed'
output_excel_path = os.path.join(input_folder, "0_Mean_Gray_Values_All_Images.xlsx")
calculate_mean_gray_all_images(input_folder, output_excel_path)

Found 945 image files
Grouped into 7 sample groups
  Replicate 01: 45 images
    Average images 30-32: 8.45
    Average last 3: 8.00
    Change in intensity: -0.44
    Percentage change: -5.25%
  Replicate 02: 45 images
    Average images 30-32: 8.34
    Average last 3: 7.00
    Change in intensity: -1.35
    Percentage change: -16.14%
  Replicate 03: 45 images
    Average images 30-32: 8.03
    Average last 3: 7.70
    Change in intensity: -0.33
    Percentage change: -4.12%
Sheet 'Co_10uMw3': 3 replicates
  Replicate 01: 45 images
    Average images 30-32: 13.67
    Average last 3: 13.57
    Change in intensity: -0.10
    Percentage change: -0.75%
  Replicate 02: 45 images
    Average images 30-32: 10.97
    Average last 3: 10.37
    Change in intensity: -0.60
    Percentage change: -5.47%
  Replicate 03: 45 images
    Average images 30-32: 7.26
    Average last 3: 7.87
    Change in intensity: 0.61
    Percentage change: 8.35%
Sheet 'Co_20uMw3': 3 replicates
  Replicate 01: 45 image

In [4]:
import os
import imageio
from natsort import natsorted

def compile_images_to_gif(input_folder):
    """
    Compiles images in a folder into GIFs based on the first four elements of the file name.
    Saves the output GIFs in a subdirectory of the input folder.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.

    Returns
    -------
    None
    """
    # Define the output folder as a subdirectory of the input folder
    output_folder = os.path.join(input_folder, "GIFs")
    os.makedirs(output_folder, exist_ok=True)

    # Dictionary to group images by the first four name elements
    grouped_images = {}

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first four name elements as the group key
            group_key = "_".join(file_name.split("_")[:4])
            if group_key not in grouped_images:
                grouped_images[group_key] = []
            grouped_images[group_key].append(file_name)

    # Process each group
    for group_key, file_names in grouped_images.items():
        # Sort the file names naturally to maintain the correct order
        file_names = natsorted(file_names)

        # List to store image data
        images = []

        # Load each image and append it to the list
        for file_name in file_names:
            image_path = os.path.join(input_folder, file_name)
            image = imageio.imread(image_path)
            images.append(image)

        # Save the images as a GIF
        gif_path = os.path.join(output_folder, f"{group_key}.gif")
        imageio.mimsave(gif_path, images, duration=0.1)  # Adjust duration as needed
        # Calculate and print the length of the GIF
        gif_length = len(images) * 0.1  # duration per frame
        print(f"Saved GIF: {gif_path} (Length: {gif_length} seconds)")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded'
compile_images_to_gif(input_folder)

/var/folders/9q/b735k4x1335_j77np9rz8xlc0000gn/T/ipykernel_83458/4081306378.py:46: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(image_path)


Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Co_Ctrl_05_C.gif (Length: 9.0 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Tri_10µMw3_5Glu_02.gif (Length: 9.0 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Co_10µMw3_25Glu_01.gif (Length: 8.700000000000001 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Tri_100µMw3_25Glu_03.gif (Length: 9.0 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Co_10µMw3_25Glu_02.gif (Length: 9.0 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Tri_Ctrl_06_C.gif (Length: 9.0 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318_Update_Ca_Image_Analysis/0_Thresholded/GIFs/Co_0w3_25Glu_01.gif (Length: 9.0 seconds)
Saved GIF: /Users/allisonpickle/Desktop/260318

In [6]:
import os
import cv2
import pandas as pd
import numpy as np
from collections import defaultdict

def calculate_masked_mean_gray_change(input_folder, output_excel_path):
    """
    Creates a mask using a mean threshold of the final image in a sequence and applies it
    to the first and last images to calculate the change in mean gray value in the masked areas only.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Dictionary to group images by the first three name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first three name elements as the group key
            group_key = "_".join(file_name.split("_")[:3])
            grouped_images[group_key].append(file_name)

    # List to store results
    results = []

    # Process each group
    for group_key, file_names in grouped_images.items():
        # Sort the file names to ensure the first and last images are correctly identified
        file_names.sort()

        # Get the first and last image in the group
        first_image_path = os.path.join(input_folder, file_names[0])
        last_image_path = os.path.join(input_folder, file_names[-1])

        # Load the first and last images
        first_image = cv2.imread(first_image_path, cv2.IMREAD_GRAYSCALE)
        last_image = cv2.imread(last_image_path, cv2.IMREAD_GRAYSCALE)

        if first_image is None or last_image is None:
            print(f"Error: Unable to load images for group {group_key}")
            continue

        # Create a mask using the mean threshold of the last image
        mean_threshold = last_image.mean()
        _, mask = cv2.threshold(last_image, mean_threshold, 255, cv2.THRESH_BINARY)

        # Apply the mask to the first and last images
        masked_first_image = cv2.bitwise_and(first_image, first_image, mask=mask.astype(np.uint8))
        masked_last_image = cv2.bitwise_and(last_image, last_image, mask=mask.astype(np.uint8))

        # Calculate the mean gray value in the masked areas
        masked_first_mean = masked_first_image.sum() / np.count_nonzero(mask) if np.count_nonzero(mask) > 0 else 0
        masked_last_mean = masked_last_image.sum() / np.count_nonzero(mask) if np.count_nonzero(mask) > 0 else 0

        # Calculate the change in intensity (difference and percentage change)
        change_in_intensity = masked_last_mean - masked_first_mean
        percentage_change = ((masked_last_mean - masked_first_mean) / masked_first_mean) * 100 if masked_first_mean != 0 else None

        # Append the results for this group
        results.append({
            "Group Key": group_key,
            "First Image": file_names[0],
            "Last Image": file_names[-1],
            "Masked First Mean Gray Value": masked_first_mean,
            "Masked Last Mean Gray Value": masked_last_mean,
            "Change in Intensity": change_in_intensity,
            "Percentage Change (%)": percentage_change
        })

    # Convert results to a DataFrame
    df = pd.DataFrame(results)

    # Save the results to an Excel file
    df.to_excel(output_excel_path, index=False)
    print(f"Results saved to Excel file: {output_excel_path}")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All'
output_excel_path = os.path.join(input_folder, "0_Masked_Mean_Gray_Values_Change.xlsx")
calculate_masked_mean_gray_change(input_folder, output_excel_path)

Results saved to Excel file: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/0_Masked_Mean_Gray_Values_Change.xlsx


In [1]:
# Average of first 3
import os
import cv2
import pandas as pd
import numpy as np
from collections import defaultdict

def calculate_masked_mean_gray_change(input_folder, output_excel_path):
    """
    Creates a mask using a mean threshold of the final image in a sequence and applies it
    to the first, last, and average of first 3 images to calculate changes in mean gray value 
    in the masked areas only.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Dictionary to group images by the first three name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first three name elements as the group key
            group_key = "_".join(file_name.split("_")[:3])
            grouped_images[group_key].append(file_name)

    # List to store results
    results = []

    # Process each group
    for group_key, file_names in grouped_images.items():
        # Sort the file names to ensure correct order
        file_names.sort()

        # Check if there are at least 3 images in the group
        if len(file_names) < 3:
            print(f"Warning: Group {group_key} has fewer than 3 images. Skipping.")
            continue

        # Get the first, last, and first 3 images
        first_image_path = os.path.join(input_folder, file_names[0])
        last_image_path = os.path.join(input_folder, file_names[-1])
        first_three_paths = [os.path.join(input_folder, fn) for fn in file_names[:3]]

        # Load the first and last images
        first_image = cv2.imread(first_image_path, cv2.IMREAD_GRAYSCALE)
        last_image = cv2.imread(last_image_path, cv2.IMREAD_GRAYSCALE)

        if first_image is None or last_image is None:
            print(f"Error: Unable to load first or last image for group {group_key}")
            continue

        # Load the first 3 images
        first_three_images = []
        for path in first_three_paths:
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"Warning: Unable to load {path}. Skipping this image.")
                continue
            first_three_images.append(img)

        if len(first_three_images) == 0:
            print(f"Error: No valid images in first 3 for group {group_key}")
            continue

        # Calculate the average of the first 3 images
        avg_first_three = np.mean(first_three_images, axis=0).astype(np.uint8)

        # Create a mask using the mean threshold of the last image
        mean_threshold = last_image.mean()
        _, mask = cv2.threshold(last_image, mean_threshold, 255, cv2.THRESH_BINARY)

        # Apply the mask to the first, last, and average images
        masked_first_image = cv2.bitwise_and(first_image, first_image, mask=mask.astype(np.uint8))
        masked_last_image = cv2.bitwise_and(last_image, last_image, mask=mask.astype(np.uint8))
        masked_avg_first_three = cv2.bitwise_and(avg_first_three, avg_first_three, mask=mask.astype(np.uint8))

        # Calculate the mean gray value in the masked areas
        mask_count = np.count_nonzero(mask)
        
        if mask_count > 0:
            masked_first_mean = masked_first_image.sum() / mask_count
            masked_last_mean = masked_last_image.sum() / mask_count
            masked_avg_first_three_mean = masked_avg_first_three.sum() / mask_count
        else:
            masked_first_mean = 0
            masked_last_mean = 0
            masked_avg_first_three_mean = 0

        # Calculate changes from first image
        change_from_first = masked_last_mean - masked_first_mean
        percentage_change_from_first = ((masked_last_mean - masked_first_mean) / masked_first_mean) * 100 if masked_first_mean != 0 else None

        # Calculate changes from average of first 3
        change_from_avg = masked_last_mean - masked_avg_first_three_mean
        percentage_change_from_avg = ((masked_last_mean - masked_avg_first_three_mean) / masked_avg_first_three_mean) * 100 if masked_avg_first_three_mean != 0 else None

        # Append the results for this group
        results.append({
            "Group Key": group_key,
            "First Image": file_names[0],
            "Last Image": file_names[-1],
            "First 3 Images": ", ".join(file_names[:3]),
            "Total Images in Sequence": len(file_names),
            "Masked First Mean Gray Value": masked_first_mean,
            "Masked Avg First 3 Mean Gray Value": masked_avg_first_three_mean,
            "Masked Last Mean Gray Value": masked_last_mean,
            "Change from First": change_from_first,
            "% Change from First": percentage_change_from_first,
            "Change from Avg First 3": change_from_avg,
            "% Change from Avg First 3": percentage_change_from_avg
        })

    # Convert results to a DataFrame
    df = pd.DataFrame(results)

    # Save the results to an Excel file
    df.to_excel(output_excel_path, index=False)
    print(f"Results saved to Excel file: {output_excel_path}")
    print(f"Processed {len(results)} image sequences")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed'
output_excel_path = os.path.join(input_folder, "0_Masked_Mean_Gray_Values_Change_first3avg.xlsx")
calculate_masked_mean_gray_change(input_folder, output_excel_path)

Results saved to Excel file: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/0_Masked_Mean_Gray_Values_Change_first3avg.xlsx
Processed 21 image sequences


In [9]:
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict

def plot_calcium_traces_by_group(input_folder, output_folder, baseline_frames=3, skip_frames=[12, 23, 34, 40], skip_last_n=4):
    """
    Plot calcium imaging traces with baseline correction (ΔF) for visual comparison.
    Groups are determined by first TWO name elements.
    Each group gets its own plot with all sequences in that group.
    Also exports an Excel file with all ΔF values.
    Measures intensity from the ENTIRE image (no masking).
    
    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_folder : str
        Path to save the plot images and Excel file.
    baseline_frames : int
        Number of initial frames to use for baseline (default: 3)
    skip_frames : list
        List of frame indices to skip (default: [12, 23, 34, 40])
    skip_last_n : int
        Number of frames to skip from the end of each sequence (default: 4)
    
    Returns
    -------
    None
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Dictionary to group images by the first TWO name elements
    grouped_images = defaultdict(lambda: defaultdict(list))

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):
            name_parts = file_name.split("_")
            if len(name_parts) < 3:
                print(f"Warning: {file_name} doesn't have enough name elements. Skipping.")
                continue
            
            # Group by first TWO elements
            group_key = "_".join(name_parts[:2])
            # Sequence within group by first THREE elements
            sequence_key = "_".join(name_parts[:3])
            
            grouped_images[group_key][sequence_key].append(file_name)

    print(f"Found {len(grouped_images)} groups")
    print(f"Skipping frames: {skip_frames}")
    print(f"Skipping last {skip_last_n} frames from each sequence")
    
    # Dictionary to store all data for Excel export
    excel_data = {}
    
    # Process each GROUP (first two name elements)
    for group_key, sequences in sorted(grouped_images.items()):
        print(f"\nProcessing group: {group_key}")
        print(f"  Contains {len(sequences)} sequences")
        
        # Create a new figure for this group
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Dictionary to store data for this group's Excel sheet
        group_data = {}
        
        # Track maximum frame count for x-axis
        max_frames = 0
        
        # Process each SEQUENCE within the group
        for sequence_key, file_names in sorted(sequences.items()):
            file_names.sort()
            
            if len(file_names) < baseline_frames:
                print(f"  Warning: Sequence {sequence_key} has fewer than {baseline_frames} images. Skipping.")
                continue

            print(f"  Processing sequence: {sequence_key} ({len(file_names)} images)")
            
            # Determine which frames to skip at the end
            total_frames = len(file_names)
            last_frame_to_process = total_frames - skip_last_n
            skip_frames_end = list(range(last_frame_to_process, total_frames))
            
            print(f"    Will skip last {skip_last_n} frames: {skip_frames_end}")

            # Load first 3 images and calculate average baseline (skip if in skip_frames)
            baseline_images = []
            for i in range(min(baseline_frames, len(file_names))):
                if i in skip_frames or i in skip_frames_end:
                    print(f"    Skipping frame {i} (in skip list)")
                    continue
                    
                img_path = os.path.join(input_folder, file_names[i])
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                
                if img is None:
                    print(f"  Warning: Unable to load {file_names[i]}")
                    continue
                
                # Calculate mean intensity of entire image
                mean_intensity = img.mean()
                baseline_images.append(mean_intensity)
            
            if len(baseline_images) == 0:
                print(f"  Warning: No valid baseline images for {sequence_key}")
                continue
            
            # Calculate baseline as mean of first N frames
            baseline = np.mean(baseline_images)
            
            # Now process all images and subtract baseline
            delta_f_values = []
            raw_intensity_values = []
            frame_numbers = []
            
            for frame_idx, file_name in enumerate(file_names):
                # Skip frames in skip list OR last N frames
                if frame_idx in skip_frames or frame_idx in skip_frames_end:
                    print(f"    Skipping frame {frame_idx}")
                    continue
                
                img_path = os.path.join(input_folder, file_name)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                
                if img is None:
                    print(f"  Warning: Unable to load {file_name}")
                    continue
                
                # Calculate mean intensity of entire image
                mean_intensity = img.mean()
                
                # Subtract baseline from each image
                delta_f = mean_intensity - baseline
                delta_f_values.append(delta_f)
                raw_intensity_values.append(mean_intensity)
                frame_numbers.append(frame_idx)
            
            if len(delta_f_values) == 0:
                print(f"  Warning: No valid frames for {sequence_key}")
                continue
            
            # Update max frames for x-axis
            max_frames = max(max_frames, max(frame_numbers) + 1)
            
            # Store data for Excel export
            group_data[f"{sequence_key}_Frame"] = frame_numbers
            group_data[f"{sequence_key}_Raw_Intensity"] = raw_intensity_values
            group_data[f"{sequence_key}_Baseline"] = [baseline] * len(delta_f_values)
            group_data[f"{sequence_key}_Delta_F"] = delta_f_values
            
            # Plot the trace for this sequence (using actual frame numbers)
            ax.plot(frame_numbers, delta_f_values, marker='o', markersize=3, 
                    label=sequence_key, alpha=0.7, linewidth=1.5)
        
        # Formatting for this group's plot
        ax.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('ΔF (intensity change from baseline)', fontsize=12, fontweight='bold')
        ax.set_title(f'Calcium Imaging Traces: {group_key}\n(Baseline = Avg of First {baseline_frames} Frames, Skipping frames {skip_frames} and last {skip_last_n} frames)', 
                     fontsize=13, fontweight='bold')
        
        # Set x-axis to show every integer frame number
        ax.set_xticks(range(max_frames))
        ax.set_xticklabels(range(max_frames))
        
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save with group name
        output_path = os.path.join(output_folder, f"{group_key}_Calcium_Traces.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  Saved plot to: {output_path}")
        
        # Store group data for Excel
        if group_data:
            excel_data[group_key] = group_data
    
    # Export all data to Excel with multiple sheets
    if excel_data:
        excel_path = os.path.join(output_folder, "Calcium_Traces_Data.xlsx")
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            for group_key, group_data in excel_data.items():
                # Convert to DataFrame (pad with NaN for unequal lengths)
                df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in group_data.items()]))
                
                # Truncate sheet name to 31 characters (Excel limit)
                sheet_name = group_key[:31]
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"\n✓ Excel file saved to: {excel_path}")
        print(f"  Contains {len(excel_data)} sheets (one per group)")
    
    print(f"\nAll plots saved to: {output_folder}")


# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed'
output_folder = os.path.join(input_folder, "Calcium_Traces_By_Group")

# Create baseline-corrected plots by group
plot_calcium_traces_by_group(input_folder, output_folder, baseline_frames=3)

Found 7 groups
Skipping frames: [12, 23, 34, 40]
Skipping last 4 frames from each sequence

Processing group: Co_10uMw3
  Contains 3 sequences
  Processing sequence: Co_10uMw3_01 (45 images)
    Will skip last 4 frames: [41, 42, 43, 44]
    Skipping frame 12
    Skipping frame 23
    Skipping frame 34
    Skipping frame 40
    Skipping frame 41
    Skipping frame 42
    Skipping frame 43
    Skipping frame 44
  Processing sequence: Co_10uMw3_02 (45 images)
    Will skip last 4 frames: [41, 42, 43, 44]
    Skipping frame 12
    Skipping frame 23
    Skipping frame 34
    Skipping frame 40
    Skipping frame 41
    Skipping frame 42
    Skipping frame 43
    Skipping frame 44
  Processing sequence: Co_10uMw3_03 (45 images)
    Will skip last 4 frames: [41, 42, 43, 44]
    Skipping frame 12
    Skipping frame 23
    Skipping frame 34
    Skipping frame 40
    Skipping frame 41
    Skipping frame 42
    Skipping frame 43
    Skipping frame 44
  Saved plot to: /Users/allisonpickle/Desktop/2

In [10]:
import matplotlib.pyplot as plt
import os
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.integrate import trapezoid

def plot_calcium_traces_individual(input_folder, output_folder, skip_frames=[12, 23, 34, 45, 56, 67, 78, 89]):
    """
    Plot calcium imaging traces (raw mean gray value) for each sequence individually.
    No baseline correction or normalization - just raw intensity values.
    Groups are determined by first THREE name elements (each sequence gets its own plot).
    Also exports an Excel file with all raw intensity values.
    Skips specific frame numbers.
    Calculates and exports AUC (Area Under Curve) for each sequence.
    
    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_folder : str
        Path to save the plot images and Excel file.
    skip_frames : list
        List of frame indices to skip (default: [12, 23, 34, 45, 56, 67, 78, 89])
    
    Returns
    -------
    None
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Dictionary to group images by the first THREE name elements (individual sequences)
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):
            name_parts = file_name.split("_")
            if len(name_parts) < 3:
                print(f"Warning: {file_name} doesn't have enough name elements. Skipping.")
                continue
            
            # Group by first THREE elements (individual sequence)
            sequence_key = "_".join(name_parts[:3])
            grouped_images[sequence_key].append(file_name)

    print(f"Found {len(grouped_images)} sequences")
    print(f"Skipping frames: {skip_frames}")
    
    # Dictionary to store all data for Excel export
    excel_data = {}
    auc_data = []
    
    # Process each SEQUENCE
    for sequence_key, file_names in sorted(grouped_images.items()):
        file_names.sort()
        
        print(f"\nProcessing sequence: {sequence_key} ({len(file_names)} images)")
        
        # Process images, skipping specified frames
        intensity_values = []
        frame_numbers = []
        skipped_frames_list = []
        
        for frame_idx, file_name in enumerate(file_names):
            # Check if this frame should be skipped
            if frame_idx in skip_frames:
                skipped_frames_list.append(frame_idx)
                print(f"  Skipping frame {frame_idx} (in skip list)")
                continue
            
            img_path = os.path.join(input_folder, file_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                print(f"  Warning: Unable to load {file_name}")
                continue
            
            # Calculate mean intensity of entire image
            mean_intensity = img.mean()
            intensity_values.append(mean_intensity)
            frame_numbers.append(frame_idx)
        
        if len(intensity_values) == 0:
            print(f"  Warning: No valid frames for {sequence_key}")
            continue
        
        # Calculate AUC using trapezoidal rule
        auc = trapezoid(intensity_values, frame_numbers)
        print(f"  AUC: {auc:.2f}")
        
        # Store AUC data
        auc_data.append({
            'Sequence': sequence_key,
            'AUC': auc,
            'Num_Frames': len(intensity_values),
            'Frames_Skipped': len(skipped_frames_list)
        })
        
        # Store data for Excel export
        excel_data[sequence_key] = {
            'Frame': frame_numbers,
            'Mean_Gray_Value': intensity_values
        }
        
        # Create individual plot for this sequence
        fig, ax = plt.subplots(figsize=(16, 7))
        
        # Plot the trace
        ax.plot(frame_numbers, intensity_values, marker='o', markersize=4, 
                color='#0068f9', linewidth=2)
        
        # Formatting
        ax.set_xlabel('Frame Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Mean Gray Value', fontsize=12, fontweight='bold')
        ax.set_title(f'{sequence_key}\nRaw Mean Intensity per Frame (AUC: {auc:.2f})', 
                     fontsize=13, fontweight='bold')
        
        # Set x-axis to show every integer frame number with readable font
        max_frame = len(file_names) - 1
        ax.set_xticks(range(max_frame + 1))
        ax.set_xticklabels(range(max_frame + 1), fontsize=8)
        
        # Also adjust y-axis tick label size for consistency
        ax.tick_params(axis='y', labelsize=9)
        
        ax.grid(True, alpha=0.3)
        
        # Add skipped frames information at bottom
        if skipped_frames_list:
            skipped_text = "Skipped frames: " + ", ".join([str(frame) for frame in skipped_frames_list])
            fig.text(0.5, 0.02, skipped_text, ha='center', fontsize=8, style='italic', color='red')
            plt.subplots_adjust(bottom=0.12)
        else:
            plt.tight_layout()
        
        # Save with sequence name
        output_path = os.path.join(output_folder, f"{sequence_key}_Raw_Trace.png")
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  Saved plot to: {output_path}")
        if skipped_frames_list:
            print(f"  Skipped {len(skipped_frames_list)} frames: {skipped_frames_list}")
    
    # Export all data to Excel with multiple sheets
    if excel_data:
        excel_path = os.path.join(output_folder, "Raw_Intensity_Data.xlsx")
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            for sequence_key, data in excel_data.items():
                # Convert to DataFrame
                df = pd.DataFrame(data)
                
                # Truncate sheet name to 31 characters (Excel limit)
                sheet_name = sequence_key[:31]
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        
        print(f"\n✓ Excel file saved to: {excel_path}")
        print(f"  Contains {len(excel_data)} sheets (one per sequence)")
    
    # Export AUC data to separate Excel file
    if auc_data:
        auc_df = pd.DataFrame(auc_data)
        auc_excel_path = os.path.join(output_folder, "AUC_Summary.xlsx")
        auc_df.to_excel(auc_excel_path, index=False)
        
        print(f"\n✓ AUC summary saved to: {auc_excel_path}")
        print(f"  Contains AUC values for {len(auc_data)} sequences")
    
    print(f"\nAll plots saved to: {output_folder}")


# Example usage
input_folder = '/Users/allisonpickle/Desktop/260226_Calcium Imaging/Plate 2/All'
output_folder = os.path.join(input_folder, "Individual_Raw_Traces")

# Create raw intensity plots for each sequence
plot_calcium_traces_individual(input_folder, output_folder, skip_frames=[12, 23, 34, 45, 56, 67, 78, 89])

Found 17 sequences
Skipping frames: [12, 23, 34, 45, 56, 67, 78, 89]

Processing sequence: Co_C1w3_01 (90 images)
  Skipping frame 12 (in skip list)
  Skipping frame 23 (in skip list)
  Skipping frame 34 (in skip list)
  Skipping frame 45 (in skip list)
  Skipping frame 56 (in skip list)
  Skipping frame 67 (in skip list)
  Skipping frame 78 (in skip list)
  Skipping frame 89 (in skip list)
  AUC: 223.83
  Saved plot to: /Users/allisonpickle/Desktop/260226_Calcium Imaging/Plate 2/All/Individual_Raw_Traces/Co_C1w3_01_Raw_Trace.png
  Skipped 8 frames: [12, 23, 34, 45, 56, 67, 78, 89]

Processing sequence: Co_C1w3_02 (90 images)
  Skipping frame 12 (in skip list)
  Skipping frame 23 (in skip list)
  Skipping frame 34 (in skip list)
  Skipping frame 45 (in skip list)
  Skipping frame 56 (in skip list)
  Skipping frame 67 (in skip list)
  Skipping frame 78 (in skip list)
  Skipping frame 89 (in skip list)
  AUC: 427.78
  Saved plot to: /Users/allisonpickle/Desktop/260226_Calcium Imaging/Pla

In [10]:
import os
import cv2
import pandas as pd
import numpy as np
from collections import defaultdict

def calculate_masked_mean_gray_change_with_visualization(input_folder, output_excel_path):
    """
    Creates a mask using the image ending in '_44' in a sequence and applies it to the first image
    (ending in '_0') and the last image (ending in '_44') to calculate the change in mean gray value
    in the masked areas only. Saves the mask as red-shaded regions overlaid on the first and last images.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Create an output folder for visualizations
    visualization_folder = os.path.join(input_folder, "Masked_Visualizations")
    os.makedirs(visualization_folder, exist_ok=True)

    # Dictionary to group images by the first three name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder
    for file_name in os.listdir(input_folder):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first three name elements as the group key
            group_key = "_".join(file_name.split("_")[:4])
            grouped_images[group_key].append(file_name)

    # List to store results
    results = []

    # Process each group
    for group_key, file_names in grouped_images.items():
        # Sort the file names naturally to ensure correct order
        file_names = sorted(file_names)

        # Identify the first image (ending in '_0') and the mask image (ending in '_44')
        first_image_path = next((os.path.join(input_folder, f) for f in file_names if f.endswith("_0.tiff")), None)
        mask_image_path = next((os.path.join(input_folder, f) for f in file_names if f.endswith("_44.tiff")), None)

        if not first_image_path or not mask_image_path:
            print(f"Skipping group {group_key}: Missing '_0' or '_44' image.")
            continue

        # Load the first image and the mask image
        first_image = cv2.imread(first_image_path, cv2.IMREAD_GRAYSCALE)
        mask_image = cv2.imread(mask_image_path, cv2.IMREAD_GRAYSCALE)

        if first_image is None or mask_image is None:
            print(f"Error: Unable to load images for group {group_key}")
            continue

        # Create a mask using the mean threshold of the mask image
        mean_threshold = mask_image.mean()
        _, mask = cv2.threshold(mask_image, mean_threshold, 255, cv2.THRESH_BINARY)

        # Apply the mask to the first and mask images
        masked_first_image = cv2.bitwise_and(first_image, first_image, mask=mask.astype(np.uint8))
        masked_mask_image = cv2.bitwise_and(mask_image, mask_image, mask=mask.astype(np.uint8))

        # Calculate the mean gray value in the masked areas
        masked_first_mean = masked_first_image.sum() / np.count_nonzero(mask) if np.count_nonzero(mask) > 0 else 0
        masked_mask_mean = masked_mask_image.sum() / np.count_nonzero(mask) if np.count_nonzero(mask) > 0 else 0

        # Calculate the change in intensity (difference and percentage change)
        change_in_intensity = masked_mask_mean - masked_first_mean
        percentage_change = ((masked_mask_mean - masked_first_mean) / masked_first_mean) * 100 if masked_first_mean != 0 else None

        # Create red-shaded visualizations for the mask
        for image, image_name, masked_image in [(first_image, "first_image", masked_first_image),
                                                (mask_image, "mask_image", masked_mask_image)]:
            # Convert the grayscale image to BGR
            image_bgr = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

            # Create a red overlay for the mask
            red_overlay = np.zeros_like(image_bgr)
            red_overlay[:, :, 2] = mask  # Set the red channel to the mask

            # Blend the original image with the red overlay
            blended_image = cv2.addWeighted(image_bgr, 0.7, red_overlay, 0.3, 0)

            # Save the visualization
            visualization_path = os.path.join(visualization_folder, f"{group_key}_{image_name}_masked.png")
            cv2.imwrite(visualization_path, blended_image)
            print(f"Saved visualization: {visualization_path}")

        # Append the results for this group
        results.append({
            "Group Key": group_key,
            "First Image": os.path.basename(first_image_path),
            "Mask Image": os.path.basename(mask_image_path),
            "Masked First Mean Gray Value": masked_first_mean,
            "Masked Mask Mean Gray Value": masked_mask_mean,
            "Change in Intensity": change_in_intensity,
            "Percentage Change (%)": percentage_change
        })

    # Convert results to a DataFrame
    df = pd.DataFrame(results)

    # Save the results to an Excel file
    df.to_excel(output_excel_path, index=False)
    print(f"Results saved to Excel file: {output_excel_path}")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed'
output_excel_path = os.path.join(input_folder, "Masked_Mean_Gray_Values_Change.xlsx")
calculate_masked_mean_gray_change_with_visualization(input_folder, output_excel_path)

Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/Tri_10uMw3_03_C_first_image_masked.png
Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/Tri_10uMw3_03_C_mask_image_masked.png
Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/Tri_Glu_01_C_first_image_masked.png
Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/Tri_Glu_01_C_mask_image_masked.png
Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/Co_20uMw3_03_C_first_image_masked.png
Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/Co_20uMw3_03_C_mask_image_masked.png
Saved visualization: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Masked_Visualizations/

Extend to measure all frames

In [21]:
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def process_images_and_save_to_excel(input_folder, output_excel_path):
    """
    Iterates through a folder of images grouped by their base names (first 2 elements of the file name),
    and calculates the mean gray value for all images in each sequence (0 to 44).
    Measures total brightness change across the entire image without masking.
    Saves results to Excel with individual plots per sheet and one overview plot.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Dictionary to group images by the first 2 name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder, sorted alphabetically
    for file_name in sorted(os.listdir(input_folder)):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first 2 name elements as the sheet key (changed from 3)
            sheet_key = "_".join(file_name.split("_")[:2])
            grouped_images[sheet_key].append(file_name)

    # Sort the groups by their keys
    grouped_images = {key: sorted(value) for key, value in grouped_images.items()}

    # Initialize a dictionary to store results for each sheet
    sheet_data = defaultdict(lambda: defaultdict(lambda: [None] * 45))  # 45 slots for sequence numbers 0 to 44

    # Process each group
    for sheet_key, file_names in grouped_images.items():
        # Process each group within the same sheet
        for file_name in file_names:
            # Extract the group key (first 4 elements) for the column name
            group_key = "_".join(file_name.split("_")[:4])

            # Load the image
            image_path = os.path.join(input_folder, file_name)
            image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

            if image is None:
                print(f"Error: Unable to load image {file_name}")
                continue

            # Extract the sequence number from the file name (e.g., '_0', '_1', ..., '_44')
            try:
                sequence_number = int(file_name.split("_")[-1].split(".")[0])
            except ValueError:
                print(f"Skipping file {file_name}: Unable to extract sequence number.")
                continue

            # Calculate the mean gray value of the entire image
            mean_gray_value = image.mean()

            # Store the result in the dictionary for the corresponding sheet and group
            sheet_data[sheet_key][group_key][sequence_number] = mean_gray_value

    # Create plots folder
    plots_folder = os.path.join(input_folder, "Plots")
    os.makedirs(plots_folder, exist_ok=True)

    # Create an Excel writer to save results in separate sheets
    with pd.ExcelWriter(output_excel_path, engine="xlsxwriter") as writer:
        # Store all averages for overview plot
        all_averages = {}
        
        for sheet_key, group_results in sheet_data.items():
            # Create a DataFrame for the current sheet
            df_data = {"Sequence Number": list(range(45))}  # First column: 0 to 44
            for group_key, values in group_results.items():
                df_data[group_key] = values

            df = pd.DataFrame(df_data)

            # Identify group columns
            group_columns = [col for col in df.columns if col != "Sequence Number"]
            
            # Calculate average across all groups
            df["Average"] = df[group_columns].mean(axis=1, skipna=True)
            
            # Store average for overview plot
            all_averages[sheet_key] = df["Average"].values
            
            # Save data
            df.to_excel(writer, index=False, sheet_name=sheet_key[:31])
            print(f"Saved results to sheet '{sheet_key[:31]}'.")
            
            # Create individual plot
            fig, ax = plt.subplots(figsize=(12, 6))
            
            # Plot each group
            for group_key in group_columns:
                ax.plot(df["Sequence Number"], df[group_key], 
                       'o-', alpha=0.5, label=group_key, linewidth=2, markersize=4)
            
            # Plot average with emphasis
            ax.plot(df["Sequence Number"], df["Average"], 
                   'k-', linewidth=3, label='Average', marker='s', markersize=5)
            
            ax.set_xlabel('Sequence Number', fontsize=12, fontweight='bold')
            ax.set_ylabel('Mean Gray Value (Brightness)', fontsize=12, fontweight='bold')
            ax.set_title(f'Brightness Change Over Time - {sheet_key}', fontsize=14, fontweight='bold')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            ax.grid(True, alpha=0.3)
            
            # Save individual plot
            plot_path = os.path.join(plots_folder, f"{sheet_key}_plot.png")
            plt.tight_layout()
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"Saved plot: {plot_path}")

    # Create overview plot with all groups
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Use a colormap to distinguish different groups
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_averages)))
    
    for (sheet_key, averages), color in zip(all_averages.items(), colors):
        ax.plot(range(45), averages, 'o-', color=color, label=sheet_key, 
               linewidth=2.5, markersize=5, alpha=0.8)
    
    ax.set_xlabel('Sequence Number', fontsize=14, fontweight='bold')
    ax.set_ylabel('Mean Gray Value (Average)', fontsize=14, fontweight='bold')
    ax.set_title('Overview: Brightness Change Over Time - All Groups', fontsize=16, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Save overview plot
    overview_plot_path = os.path.join(plots_folder, "Overview_All_Groups.png")
    plt.tight_layout()
    plt.savefig(overview_plot_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved overview plot: {overview_plot_path}")

    print(f"\n{'='*60}")
    print(f"Final results saved to Excel file: {output_excel_path}")
    print(f"All plots saved to: {plots_folder}")
    print(f"{'='*60}")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed'
output_excel_path = '/Users/allisonpickle/Desktop/All_Images_Brightness.xlsx'
process_images_and_save_to_excel(input_folder, output_excel_path)

Saved results to sheet 'Co_10uMw3'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_10uMw3_plot.png
Saved results to sheet 'Co_20uMw3'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_20uMw3_plot.png
Saved results to sheet 'Co_Ctrl'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_Ctrl_plot.png
Saved results to sheet 'Tri_10uMw3'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Tri_10uMw3_plot.png
Saved results to sheet 'Tri_20uMw3'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Tri_20uMw3_plot.png
Saved results to sheet 'Tri_Ctrl'.


Exception ignored in: <function ZipFile.__del__ at 0x10813d0d0>
Traceback (most recent call last):
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/zipfile.py", line 1807, in __del__
    self.close()
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/zipfile.py", line 1824, in close
    self.fp.seek(self.start_dir)
TimeoutError: [Errno 60] Operation timed out


Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Tri_Ctrl_plot.png
Saved results to sheet 'Tri_Glu'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Tri_Glu_plot.png
Saved overview plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Overview_All_Groups.png

Final results saved to Excel file: /Users/allisonpickle/Desktop/All_Images_Brightness.xlsx
All plots saved to: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots


In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def process_images_and_save_to_excel(input_folder, output_excel_path):
    """
    Iterates through a folder of images grouped by their base names (first 4 elements of the file name),
    and calculates the mean gray value for all images in each sequence (0 to 44).
    Measures total brightness change across the entire image without masking.
    Saves results to Excel with individual plots per sheet and one overview plot.

    Parameters
    ----------
    input_folder : str
        Path to the folder containing the images.
    output_excel_path : str
        Path to save the Excel file with the results.

    Returns
    -------
    None
    """
    # Dictionary to group images by the first 3 name elements
    grouped_images = defaultdict(list)

    # Iterate through all files in the folder, sorted alphabetically
    for file_name in sorted(os.listdir(input_folder)):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):  # Check for valid image formats
            # Extract the first 3 name elements as the sheet key
            sheet_key = "_".join(file_name.split("_")[:3])
            grouped_images[sheet_key].append(file_name)

    # Sort the groups by their keys
    grouped_images = {key: sorted(value) for key, value in grouped_images.items()}

    # Initialize a dictionary to store results for each sheet
    sheet_data = defaultdict(lambda: defaultdict(lambda: [None] * 45))  # 45 slots for sequence numbers 0 to 44

    # Process each group
    for sheet_key, file_names in grouped_images.items():
        # Process each group within the same sheet
        for file_name in file_names:
            # Extract the group key (first 4 elements) for the column name
            group_key = "_".join(file_name.split("_")[:4])

            # Load the image
            image_path = os.path.join(input_folder, file_name)
            image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

            if image is None:
                print(f"Error: Unable to load image {file_name}")
                continue

            # Extract the sequence number from the file name (e.g., '_0', '_1', ..., '_44')
            try:
                sequence_number = int(file_name.split("_")[-1].split(".")[0])
            except ValueError:
                print(f"Skipping file {file_name}: Unable to extract sequence number.")
                continue

            # Calculate the mean gray value of the entire image
            mean_gray_value = image.mean()

            # Store the result in the dictionary for the corresponding sheet and group
            sheet_data[sheet_key][group_key][sequence_number] = mean_gray_value

    # Create plots folder
    plots_folder = os.path.join(input_folder, "Plots")
    os.makedirs(plots_folder, exist_ok=True)

    # Create an Excel writer to save results in separate sheets
    with pd.ExcelWriter(output_excel_path, engine="xlsxwriter") as writer:
        # Store all averages for overview plot
        all_averages = {}
        
        for sheet_key, group_results in sheet_data.items():
            # Create a DataFrame for the current sheet
            df_data = {"Sequence Number": list(range(45))}  # First column: 0 to 44
            for group_key, values in group_results.items():
                df_data[group_key] = values

            df = pd.DataFrame(df_data)

            # Identify group columns
            group_columns = [col for col in df.columns if col != "Sequence Number"]
            
            # Calculate average across all groups
            df["Average"] = df[group_columns].mean(axis=1, skipna=True)
            
            # Store average for overview plot
            all_averages[sheet_key] = df["Average"].values
            
            # Save data
            df.to_excel(writer, index=False, sheet_name=sheet_key[:31])
            print(f"Saved results to sheet '{sheet_key[:31]}'.")
            
            # Create individual plot
            fig, ax = plt.subplots(figsize=(12, 6))
            
            # Plot each group
            for group_key in group_columns:
                ax.plot(df["Sequence Number"], df[group_key], 
                       'o-', alpha=0.5, label=group_key, linewidth=2, markersize=4)
            
            # Plot average with emphasis
            ax.plot(df["Sequence Number"], df["Average"], 
                   'k-', linewidth=3, label='Average', marker='s', markersize=5)
            
            ax.set_xlabel('Sequence Number', fontsize=12, fontweight='bold')
            ax.set_ylabel('Mean Gray Value (Brightness)', fontsize=12, fontweight='bold')
            ax.set_title(f'Brightness Change Over Time - {sheet_key}', fontsize=14, fontweight='bold')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            ax.grid(True, alpha=0.3)
            
            # Save individual plot
            plot_path = os.path.join(plots_folder, f"{sheet_key}_plot.png")
            plt.tight_layout()
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"Saved plot: {plot_path}")

    # Create overview plot with all groups
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Use a colormap to distinguish different groups
    colors = plt.cm.tab20(np.linspace(0, 1, len(all_averages)))
    
    for (sheet_key, averages), color in zip(all_averages.items(), colors):
        ax.plot(range(45), averages, 'o-', color=color, label=sheet_key, 
               linewidth=2.5, markersize=5, alpha=0.8)
    
    ax.set_xlabel('Sequence Number', fontsize=14, fontweight='bold')
    ax.set_ylabel('Mean Gray Value (Average)', fontsize=14, fontweight='bold')
    ax.set_title('Overview: Brightness Change Over Time - All Groups', fontsize=16, fontweight='bold')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Save overview plot
    overview_plot_path = os.path.join(plots_folder, "Overview_All_Groups.png")
    plt.tight_layout()
    plt.savefig(overview_plot_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved overview plot: {overview_plot_path}")

    print(f"\n{'='*60}")
    print(f"Final results saved to Excel file: {output_excel_path}")
    print(f"All plots saved to: {plots_folder}")
    print(f"{'='*60}")

# Example usage
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed'
output_excel_path = '/Users/allisonpickle/Desktop/All_Images_Brightness.xlsx'
process_images_and_save_to_excel(input_folder, output_excel_path)

Saved results to sheet 'Co_10uMw3_01'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_10uMw3_01_plot.png
Saved results to sheet 'Co_10uMw3_02'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_10uMw3_02_plot.png
Saved results to sheet 'Co_10uMw3_03'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_10uMw3_03_plot.png
Saved results to sheet 'Co_20uMw3_01'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_20uMw3_01_plot.png
Saved results to sheet 'Co_20uMw3_02'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_20uMw3_02_plot.png
Saved results to sheet 'Co_20uMw3_03'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/Plots/Co_20uMw3_03_plot.png
Saved results to sheet 'Co_Ctrl_01'.
Saved plot: /Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed/P

In [20]:
import os
from collections import defaultdict

def diagnose_grouping(input_folder):
    """
    Diagnoses how files are being grouped to identify duplicates.
    """
    grouped_images = defaultdict(list)
    
# Iterate through all files in the folder, sorted alphabetically
    for file_name in sorted(os.listdir(input_folder)):
        if file_name.endswith((".tiff", ".tif", ".png", ".jpg", ".jpeg")):
            # Extract the first 2 name elements as the sheet key (changed from 3)
            sheet_key = "_".join(file_name.split("_")[:2])
            grouped_images[sheet_key].append(file_name)

    
    # Analyze each sheet
    for sheet_key, files in grouped_images.items():
        print(f"\n{'='*60}")
        print(f"Sheet: {sheet_key}")
        print(f"{'='*60}")
        
        # Group by group_key to find duplicates
        group_dict = defaultdict(list)
        for file_name, group_key in files:
            group_dict[group_key].append(file_name)
        
        for group_key, file_list in group_dict.items():
            print(f"\nGroup Key: {group_key}")
            print(f"  Number of files: {len(file_list)}")
            if len(file_list) <= 5:  # Show first 5 files
                for f in file_list:
                    print(f"    - {f}")
            else:
                for f in file_list[:3]:
                    print(f"    - {f}")
                print(f"    ... and {len(file_list) - 3} more files")

# Run diagnostic
input_folder = '/Users/allisonpickle/Desktop/260122_Calcium Imaging_w3/All/Reversed'
diagnose_grouping(input_folder)


Sheet: Co_10uMw3


ValueError: too many values to unpack (expected 2)